In [1]:
import matplotlib
import platform
print ("Operating system: ", platform.system())
if "Linux" in platform.system():
    %matplotlib tk
else:
    %matplotlib qt
    
#
import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

#
import scipy
import os
import time

#
from calibration.calibration import BMICalibration, get_binary_std_map, get_rois_stardist2d, get_img_std
from drift.drift import (make_template, compute_drift_multi_frames, correct_drift, 
                         correct_drift_single_frame, template_generation, 
                         plot_mean_vs_template, make_motion_template_and_correct_data)
from utils.utils import smooth_ca_time_series, compute_dff0



Operating system:  Windows


Autosaving every 180 seconds


auto.py (22): IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [2]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################
fname = r'D:\bmi\bmi_test\data\Image_001_001.raw'
#fname = '/media/cat/4TB/donato/DON-009460/22-06-21/calibration/Image_001_001.raw'
#fname = '/media/cat/4TB/donato/DON-008499/mouse2_calibration/Image_001_001.raw'
#fname = '/media/cat/4TBSSD/donato/test/data/Image_001_001.raw'
# 
bmi_c = BMICalibration(fname)

bmi_c.smooth_ca_time_series = smooth_ca_time_series
bmi_c.compute_dff0 = compute_dff0

#
bmi_c.subsample = 10 # for std computation downsample to every N'th frame; the more frames the better the rois;
                  #   TODO: use correlation instead?! might be much faster; it is quit fast in other implemenations


memmap :  (27000, 512, 512)


In [3]:
##################################################################
############### MOTION CORRECTION STEP ###########################
##################################################################
# 
start = time.time()
bmi_c = make_motion_template_and_correct_data(bmi_c)
plot_mean_vs_template(bmi_c)
print ("total processing time: ", time.time()-start, " sec")
print ("DONE...")

100%|█████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:11<00:00,  3.35it/s]


computing motion on:  D:\bmi\bmi_test\data\Image_001_001.raw


100%|█████████████████████████████████████████████████████████████████████████████████████| 40/40 [01:41<00:00,  2.54s/it]


TODO: undo interpolation for drift with better function


100%|█████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:12<00:00,  3.27it/s]


(27000, 512, 512) (1000,)
total processing time:  151.67949867248535  sec
DONE...


In [4]:
#######################################################################
##################### GET STD MAP TO GET CELL FOOTPRINTS ##############
#######################################################################
# 
start = time.time()
std_map = bmi_c.make_std_map()
#std_map = bmi_c.make_max_proj_map()
print ("total processing time: ", time.time()-start, " sec")
print ("...DONE...")


data into analysis:  (2700, 512, 512)
 gaussian filter width:  1 , order:  0
done filtering... (TO CHECK which axis are we filtering!!)
total processing time:  30.277496099472046  sec
...DONE...


In [6]:
#################################################################
########### MAKE BINARY MAP FOR CELL REGISTRATOIN ###############
#################################################################

# get binary map
vmax = 500
smin, smax = get_binary_std_map(std_map,
                                        vmax)



In [7]:
##################################################

bmi_c, img_std = get_img_std(smin, smax, std_map, bmi_c)


max proj values (vmin, vmax):  92.75 93.75


In [8]:
####################################################
####################################################
####################################################
bmi_c.rois, bmi_c.indexes = get_rois_stardist2d(img_std)

There are 4 registered models for 'StarDist2D':

Name                  Alias(es)
────                  ─────────
'2D_versatile_fluo'   'Versatile (fluorescent nuclei)'
'2D_versatile_he'     'Versatile (H&E nuclei)'
'2D_paper_dsb2018'    'DSB 2018 (from StarDist 2D paper)'
'2D_demo'             None
None
Found model '2D_versatile_fluo' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479071, nms_thresh=0.3.
1/1 [==============================] - 0s 277ms/step


looping over cells: 100%|███████████████████████████████████████████████████████████████| 79/79 [00:00<00:00, 1553.18it/s]


In [11]:
####################################################
############## SELECT ENSEMBLE CELLS ###############
####################################################
#
bmi_c.scale=3000                      # spacing between each neuron trace because they are not normalized to the max vlaue
bmi_c.trace_subsample = 10             # Subsample the time series to go faster;

# visualize traces
bmi_c.compute_traces2(std_map)

plotting cells:  [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73 74 75 76]
memmap :  (27000, 512, 512)


100%|████████████████████████████████████████████████████████████████████████████████| 2700/2700 [00:04<00:00, 559.34it/s]


[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73 74 75 76]


In [14]:
###############################################################
########### SELECT CELLS TO BE USED FOR ENSEMBLES #############
###############################################################

# save ensemble rois
bmi_c.ensemble1 = [1,0]
bmi_c.ensemble2 = [33,43]
both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
print ("all cells:", both)

#
bmi_c.show_traces_ids(both)


all cells: [ 1  0 33 43]


### COMPUTE THE MIN AND MAXES FOR THE SELECTED ENSMEMLES

Some important points:

1. For now we are working in pixel absolute values for each cell.  

2. A better option might be to find the maximum peak of a cell during a window and then save that value and normalize all future events by that value. (note: any online BMI filtering/chagnig of data will need to account for this).


In [15]:
# ######################################################################
# ########### RECOMPUTE TRACES WITH SINGLE FRAME PRECISION #############
# ######################################################################
bmi_c.trace_subsample = 1        # Subsample the time series to go faster;

# visualize traces
bmi_c.compute_traces2(std_map, both)

print ("DONE...")

plotting cells:  [ 1  0 33 43]
memmap :  (27000, 512, 512)


100%|█████████████████████████████████████████████████████████████████████████████| 27000/27000 [00:03<00:00, 7579.00it/s]


[ 1  0 33 43]
DONE...


In [16]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################
# 
bmi_c.sample_rate = 30
bmi_c.post_reward_lockout = 10   # reward lockout in seconds
bmi_c.balance_ensemble_rewards_flag = True   #this makes sure that both ensembles elicit a similar number of random rewards
bmi_c.rois_smooth_window = 10     # of frames to use to smooth the realtime signal
bmi_c.smooth_diff_function_flag = True    # use a kernel window to smooth current value

# find 30% reward threshold
# We have 3 options: 
#   1) rewarding  E1-E2 going above some threshold
#   2) rewarding E2-E1 going above some threhosld
#   3) rewarding both
# The challenge is mapping the ensemble states to tone/stimulus space
# 
#gr.find_reward_thresholds_low_and_high()
#gr.find_reward_thresholds_high()  # this only rewards when sound passes specific level

# this option rewards both ensembles 
normalize_peaks = True   # this flag normalizes the peaks to make sure one ensembel
                         # doesn't completely dominate the other
bmi_c.find_reward_thresholds_absolute(normalize_peaks)


#
print ("thresholds: ", bmi_c.high)

########################################
bmi_c.plot_rewarded_ensembles()

print (bmi_c.high)
bmi_c.high = bmi_c.high*1

100%|███████████████████████████████████████████████████████████████████████████| 26990/26990 [00:00<00:00, 119729.78it/s]


TODO: Normalize the peaks of the 2 Ensembles so the mice don't learn to use one esnemble against the other!!!!
low, high:  nan 6.89094798812799
nsec recording:  900 max # of random rewards (i.e. every 30sec)  30 # of rewards for 30% of the time:  9
updated rwards #:  13 nan 1.911477120831582
thresholds:  1.911477120831582
1.911477120831582


In [17]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################

# save all data to disk
# also add the tone values here as well that will be used for the experiment
bmi_c.low_freq = 2000
bmi_c.high_freq = 18000

# save cell pixel footprints locations as 2 column array inside list
cells = []
for k in range(4):
    temp = bmi_c.indexes[both[k]]
    temp1 = temp[0]
    temp2 = temp[1]
    temp = np.vstack((temp1,temp2))
    cells.append(temp)

# also grab contours of cells; both contains all cell ids
contours = bmi_c.compute_contour_map(std_map, both)
print ("len: ", len(contours))        

# save individual pixels of each cell - currently implemented in BMI
np.savez(os.path.join(os.path.split(os.path.split(fname)[0])[0],
                        'rois_pixels_and_thresholds.npz'),
            cell0_footprint = cells[0],
            cell1_footprint = cells[1],
            cell2_footprint = cells[2],
            cell3_footprint = cells[3],
            
            #
            cell0_contour = contours[0],
            cell1_contour = contours[1],
            cell2_contour = contours[2],
            cell3_contour = contours[3],
         
            #
            cell_f0s = bmi_c.roi_f0s,
            cell_centres = np.int32(bmi_c.rois)[both],
            cell_ids = both,
            all_rois = np.int32(bmi_c.rois),
            low_threshold = bmi_c.low,
            high_threshold = bmi_c.high,
            low_freq = bmi_c.low_freq,
            high_freq = bmi_c.high_freq,
            cell_traces = bmi_c.roi_traces,
            sample_rate = bmi_c.sample_rate,
            post_reward_lockout = bmi_c.post_reward_lockout,
            balance_ensemble_rewards_flag = bmi_c.balance_ensemble_rewards_flag,
            rois_smooth_window = bmi_c.rois_smooth_window,
            smooth_diff_function_flag = bmi_c.smooth_diff_function_flag,
            calibration_template = bmi_c.template,
        )



len:  4


In [19]:
data = np.load('D:\bmi\DON8460\22-07-13\databmi_results.npz',allow_pickle=True)

r1 = data['rotary_encoder1_abstime']
r2 = data['rotary_encoder2_abstime']
plt.figure()
plt.plot(r1)
plt.plot(r2)
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'D:\x08mi\\DON8460\x12-07-13\\databmi_results.npz'

In [28]:
data = np.load(r'E:\bmi_backup\DON8460\22-07-13\databmi_results.npz')
ensembles = data['ensemble_activity']

temp = data['ttl_voltages']

print (ensembles.shape)
plt.figure()
plt.plot(temp)
#plt.plot(ensembles[0])
#plt.plot(ensembles[1])
plt.show()

(2, 1000)


In [35]:
#
print (gr.diff.shape)
plt.figure(0)
plt.plot(gr.diff)
plt.show()

(20000,)


In [ ]:
#######################################################
######### MANUAL ROI SELECTOR - DO NOT DELETE #########
#######################################################

# # importing the module
# import cv2

# # function to display the coordinates of
# # of the points clicked on the image
# def click_event(event, x, y, flags, params):

#     # checking for left mouse clicks
#     if event == cv2.EVENT_LBUTTONDOWN:

#         # displaying the coordinates
#         # on the Shell
#         print(x, ' ', y)
#         rois_manual.append([x,y])

#         # displaying the coordinates
#         # on the image window
#         #font = cv2.FONT_HERSHEY_SIMPLEX
#         img[y-2:y+3, x-2:x+3] = 0
   
#         #cv2.putText(img, str(x) + ',' +
#         #            str(y), (x,y), font,
#         #            .3, (255, 0, 0), 2)
#         cv2.imshow('image', img)

#     # checking for right mouse clicks	
#     #if event==cv2.EVENT_RBUTTONDOWN:
#     #    
#     #    np.save()

# # driver function
# if __name__=="__main__":
    
#     global rois_manual
    
#     rois_manual = []
    
#     # reading the image
#     #img = cv2.imread('lena.jpg', 1)
#     img = std_map.copy()
    
#     img = cv2.resize(img, (int(img.shape[0]*1.5),
#                            int(img.shape[1]*1.5))) 

#     # displaying the image
#     cv2.imshow('image', img)

#     # setting mouse handler for the image
#     # and calling the click_event() function
#     cv2.setMouseCallback('image', click_event)

#     # wait for a key to be pressed to exit
#     cv2.waitKey(0)

#     # close the window
#     cv2.destroyAllWindows()

# print (" DONE LABELING: ")
# print ("ROIS: ", rois_manual)



In [2]:
data = np.load('/media/cat/4TB/donato/BSCOPE_tests/rois_pixels_and_thresholds.npz')

low_thresh = data['low_threshold']
high_thresh = data['high_threshold']

print (low_thresh, high_thresh)



-768.7697339352011 546.5230278373468


In [10]:

octave_size = 0.25

# 
def get_octave_frequencies(low_freq,
                  high_freq,
                  octave_size):
    
    #
    octaves = []
    
    #
    octaves.append(low_freq)
    temp = low_freq
    while True:
        temp = temp * (1+octave_size)
        if temp>high_freq:
            break
        octaves.append(temp)

    return octaves
#
octaves = get_octave_frequencies(2000,
              18000,
              octave_size)
print (len(octaves),octaves)
      
    


10 [2000, 2500.0, 3125.0, 3906.25, 4882.8125, 6103.515625, 7629.39453125, 9536.7431640625, 11920.928955078125, 14901.161193847656]
